# Notebook 04 — Real Data: Criteo Uplift Modeling Dataset

We now apply the **identical pipeline** from Notebook 03 to the Criteo Uplift Modeling Dataset — a real, research-grade randomized trial used to benchmark causal ML papers.

### About the Criteo Dataset
| Property | Value |
|---|---|
| Source | Criteo AI Lab (Kaggle) |
| Rows | ~13.4 million users |
| Features | 12 anonymized numeric features (f0–f11) |
| Treatment | Binary (ad exposure, ~85% treatment / 15% control) |
| Outcomes | `visit` (binary), `conversion` (binary) |
| Ground truth ITE | **Not available** (real experiment) |

### Key difference from synthetic data
Without ground-truth ITE, we validate using:
- **Qini curve** vs. random baseline (still meaningful)
- **AUUC** (Area Under Uplift Curve)
- **Business interpretation**: how much lift do we capture by targeting vs. random?

### Treatment imbalance note
The Criteo dataset has a ~85/15 treatment/control split by design.  
This is a great talking point: **X-Learner is specifically designed to handle this imbalance** better than T-Learner.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from scipy.stats import spearmanr
import joblib, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

SEED = 42
rng  = np.random.default_rng(SEED)
Path('figures').mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#0f1117',
    'axes.edgecolor': '#30363d',   'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e',      'ytick.color': '#8b949e',
    'text.color': '#e6edf3',       'grid.color': '#21262d',
    'grid.linestyle': '--',        'font.size': 11,
})
PALETTE = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657', '#79c0ff']

# Copy in the meta-learner classes from Notebook 03
class SLearner:
    def __init__(self, base_model): self.model = base_model
    def fit(self, X, T, Y):
        self.model.fit(np.column_stack([X, T]), Y); return self
    def predict(self, X):
        n = len(X)
        return (self.model.predict_proba(np.column_stack([X, np.ones(n)]))[:, 1] -
                self.model.predict_proba(np.column_stack([X, np.zeros(n)]))[:, 1])

class TLearner:
    def __init__(self, cls, **kw): self.m1 = cls(**kw); self.m0 = cls(**kw)
    def fit(self, X, T, Y):
        self.m1.fit(X[T==1], Y[T==1]); self.m0.fit(X[T==0], Y[T==0]); return self
    def predict(self, X):
        return self.m1.predict_proba(X)[:,1] - self.m0.predict_proba(X)[:,1]

class XLearner:
    def __init__(self, cls, **kw):
        self.m1=cls(**kw); self.m0=cls(**kw); self.mx1=cls(**kw); self.mx0=cls(**kw)
        self.ps=LogisticRegression(max_iter=500)
    def fit(self, X, T, Y):
        self.m1.fit(X[T==1],Y[T==1]); self.m0.fit(X[T==0],Y[T==0])
        tau1 = Y[T==1] - self.m0.predict_proba(X[T==1])[:,1]
        tau0 = self.m1.predict_proba(X[T==0])[:,1] - Y[T==0]
        self.mx1.fit(X[T==1],tau1); self.mx0.fit(X[T==0],tau0)
        self.ps.fit(X,T); return self
    def predict(self, X):
        e = self.ps.predict_proba(X)[:,1]
        return e * self.mx0.predict(X) + (1-e) * self.mx1.predict(X)

def compute_qini(y, treatment, uplift_score, resolution=100):
    df_q = pd.DataFrame({'y':y,'T':treatment,'score':uplift_score})
    df_q = df_q.sort_values('score', ascending=False).reset_index(drop=True)
    N = len(df_q); pcts=[0]; qini=[0.0]
    for pct in np.linspace(100/resolution, 100, resolution):
        n_tar = max(1, int(N*pct/100))
        s = df_q.iloc[:n_tar]
        nt=s['T'].sum(); nc=(1-s['T']).sum()
        if nt==0 or nc==0: qini.append(qini[-1]); pcts.append(pct); continue
        qini.append(s.loc[s.T==1,'y'].sum() - s.loc[s.T==0,'y'].sum()*(nt/nc))
        pcts.append(pct)
    return np.array(pcts), np.array(qini)

print('Setup complete ✓')

---
## Part 1 — Load the Criteo Dataset

**To download:**
```bash
# Option A: Kaggle CLI (requires Kaggle API key at ~/.kaggle/kaggle.json)
pip install kaggle
kaggle datasets download -d arashnic/criteo-uplift-prediction -p data/ --unzip

# Option B: Manual download
# Go to https://www.kaggle.com/datasets/arashnic/criteo-uplift-prediction
# Download criteo-uplift-v2.1.csv and place in data/

# Option C: On Kaggle Notebooks (no download needed — data is auto-mounted)
# Change DATA_PATH below to '/kaggle/input/criteo-uplift-prediction/criteo-uplift-v2.1.csv'
```

In [ ]:
import os

# ── Data path — update this for your environment ───────────────────────────────
CRITEO_PATHS = [
    'data/criteo-uplift-v2.1.csv',
    'data/criteo_uplift.csv',
    '/kaggle/input/criteo-uplift-prediction/criteo-uplift-v2.1.csv',
]

DATA_PATH = None
for p in CRITEO_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    print('⚠️  Criteo CSV not found. Run the download commands above.')
    print('   For now, we will generate a structural replica for code demonstration.')
    USE_SYNTHETIC_REPLICA = True
else:
    print(f'Found Criteo data at: {DATA_PATH}')
    USE_SYNTHETIC_REPLICA = False

In [ ]:
if USE_SYNTHETIC_REPLICA:
    # ── Structural replica: same schema, ~500K rows for fast demonstration ────
    print('Generating Criteo structural replica (same schema, anonymized features)...')
    N_REPLICA = 500_000
    rng2 = np.random.default_rng(SEED)

    # Treatment: ~85% treatment (matching Criteo's real imbalance)
    treatment_cr = rng2.choice([0, 1], size=N_REPLICA, p=[0.15, 0.85])

    # 12 anonymized numeric features
    features = {f'f{i}': rng2.normal(size=N_REPLICA) for i in range(12)}

    # Outcomes with realistic rates
    # Uplift is heterogeneous by f0 and f3 (arbitrary — for demonstration)
    base_visit  = 0.04 + 0.03 * (features['f0'] > 0)
    base_conv   = 0.002 + 0.001 * (features['f3'] > 0)
    treat_boost_v = 0.005 + 0.008 * (features['f0'] > 0.5) + 0.003 * (features['f2'] > 0)
    treat_boost_c = 0.0005 + 0.001 * (features['f3'] > 0.5)

    p_visit = np.clip(base_visit + treatment_cr * treat_boost_v, 0, 1)
    p_conv  = np.clip(base_conv  + treatment_cr * treat_boost_c, 0, 1)

    df_cr = pd.DataFrame(features)
    df_cr['treatment']  = treatment_cr
    df_cr['visit']      = rng2.binomial(1, p_visit)
    df_cr['conversion'] = rng2.binomial(1, p_conv)

    print(f'Replica created: {len(df_cr):,} rows, {len(df_cr.columns)} cols')
    print('NOTE: Use the real Criteo CSV for final portfolio results.')

else:
    # Load real Criteo data (sample 1M rows for speed on CPU)
    print(f'Loading Criteo data from {DATA_PATH}...')
    df_cr = pd.read_csv(DATA_PATH, nrows=1_000_000)
    print(f'Loaded {len(df_cr):,} rows')

print(f'Columns: {df_cr.columns.tolist()}')
df_cr.head(3)

---
## Part 2 — Exploratory Data Analysis

In [ ]:
feat_cols = [c for c in df_cr.columns if c.startswith('f')]
print(f'Feature columns: {feat_cols}')
print(f'\nTreatment distribution:')
print(df_cr['treatment'].value_counts(normalize=True).rename({0:'control',1:'treatment'}).round(4))
print(f'\nVisit rate (control):       {df_cr.loc[df_cr.treatment==0,"visit"].mean():.4f}')
print(f'Visit rate (treatment):     {df_cr.loc[df_cr.treatment==1,"visit"].mean():.4f}')
print(f'Conversion rate (control):  {df_cr.loc[df_cr.treatment==0,"conversion"].mean():.5f}')
print(f'Conversion rate (treatment):{df_cr.loc[df_cr.treatment==1,"conversion"].mean():.5f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(feat_cols[:6]):
    ax = axes[i]
    for grp, label, color in [(0,'Control',PALETTE[0]),(1,'Treatment',PALETTE[1])]:
        data = df_cr.loc[df_cr.treatment==grp, feat].dropna()
        ax.hist(data.clip(-4, 4), bins=60, alpha=0.6, label=label, color=color,
                density=True, edgecolor='none')
    ax.set_title(feat, fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)

plt.suptitle('Feature Distributions: Treatment vs. Control\n(Good balance = overlapping histograms)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/04_criteo_feature_distributions.png', dpi=130, bbox_inches='tight', facecolor='#0f1117')
plt.show()

---
## Part 3 — Uplift Modeling on Criteo

In [ ]:
# Use visit as the primary outcome (higher rate → more stable estimates)
X_cr = df_cr[feat_cols].fillna(0).values
T_cr = df_cr['treatment'].values
Y_cr = df_cr['visit'].values

# Sample 200K for tractable training time (use full data on GPU/Kaggle)
SAMPLE_SIZE = min(200_000, len(df_cr))
sample_idx  = rng.choice(len(df_cr), size=SAMPLE_SIZE, replace=False)
X_s = X_cr[sample_idx]; T_s = T_cr[sample_idx]; Y_s = Y_cr[sample_idx]

idx_train_cr, idx_test_cr = train_test_split(
    np.arange(SAMPLE_SIZE), test_size=0.25, random_state=SEED, stratify=T_s
)
X_tr_cr, X_te_cr = X_s[idx_train_cr], X_s[idx_test_cr]
T_tr_cr, T_te_cr = T_s[idx_train_cr], T_s[idx_test_cr]
Y_tr_cr, Y_te_cr = Y_s[idx_train_cr], Y_s[idx_test_cr]

print(f'Training on {len(X_tr_cr):,} samples  |  Testing on {len(X_te_cr):,} samples')
print(f'Treatment rate in train: {T_tr_cr.mean():.3f}  |  test: {T_te_cr.mean():.3f}')

In [ ]:
BASE = dict(n_estimators=150, max_depth=5, random_state=SEED, n_jobs=-1)

print('Training S-Learner on Criteo...')
sl_cr = SLearner(RandomForestClassifier(**BASE))
sl_cr.fit(X_tr_cr, T_tr_cr, Y_tr_cr)
ite_s_cr = sl_cr.predict(X_te_cr)

print('Training T-Learner on Criteo...')
tl_cr = TLearner(RandomForestClassifier, **BASE)
tl_cr.fit(X_tr_cr, T_tr_cr, Y_tr_cr)
ite_t_cr = tl_cr.predict(X_te_cr)

print('Training X-Learner on Criteo...')
xl_cr = XLearner(RandomForestClassifier, **BASE)
xl_cr.fit(X_tr_cr, T_tr_cr, Y_tr_cr)
ite_x_cr = xl_cr.predict(X_te_cr)

print('\nDone. ITE ranges:')
for name, preds in [('S-Learner', ite_s_cr), ('T-Learner', ite_t_cr), ('X-Learner', ite_x_cr)]:
    print(f'  {name}: [{preds.min():.4f}, {preds.max():.4f}]  mean={preds.mean():.4f}')

In [ ]:
# ── Qini Curves on Criteo ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

models_cr = {'S-Learner': ite_s_cr, 'T-Learner': ite_t_cr, 'X-Learner': ite_x_cr}
for (name, preds), color in zip(models_cr.items(), PALETTE):
    pcts, qini = compute_qini(Y_te_cr, T_te_cr, preds)
    auc = np.trapz(qini, pcts/100)
    ax.plot(pcts, qini, label=f'{name} (AUC={auc:.1f})', color=color, lw=2.5)

# Random baseline
pcts_r, qini_r = compute_qini(Y_te_cr, T_te_cr, rng.random(len(Y_te_cr)))
ax.plot(pcts_r, qini_r, color='#8b949e', lw=1.5, linestyle='--', label='Random baseline')

ax.set_xlabel('Population targeted (%)', fontsize=12)
ax.set_ylabel('Incremental visits', fontsize=12)
ax.set_title('Qini Curves — Criteo Uplift Dataset\n'
             '(Real-world experiment, no ground-truth ITE)', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('figures/04_criteo_qini_curves.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

---
## Part 4 — Business Translation

In [ ]:
# Best model on Criteo
best_preds_cr = ite_x_cr  # X-Learner typically best for imbalanced treatment
sorted_idx_cr = np.argsort(best_preds_cr)[::-1]

# Estimate total incremental visits in test set (imperfect — used for illustration)
# Incremental = visit_T * (1/p_treat) - visit_C * (1/p_control)  [Horvitz-Thompson estimator]
p_treat_cr = T_te_cr.mean()
p_ctrl_cr  = 1 - p_treat_cr

total_incr_est = (
    Y_te_cr[T_te_cr == 1].sum() / p_treat_cr -
    Y_te_cr[T_te_cr == 0].sum() / p_ctrl_cr
)
print(f'Estimated total incremental visits in test set: {total_incr_est:,.0f}')

# Gain at various thresholds
for pct in [10, 20, 30, 50]:
    n_tar = int(len(sorted_idx_cr) * pct / 100)
    top   = sorted_idx_cr[:n_tar]
    incr_top = (
        Y_te_cr[top][T_te_cr[top]==1].sum() / p_treat_cr -
        Y_te_cr[top][T_te_cr[top]==0].sum() / p_ctrl_cr
    ) if (T_te_cr[top]==0).sum() > 0 else 0
    pct_lift = incr_top / total_incr_est * 100 if total_incr_est != 0 else 0
    eff      = pct_lift / pct
    print(f'  Target top {pct:2d}% → captures ~{pct_lift:.1f}% of lift  ({eff:.2f}× more efficient than random)')

---
## Part 5 — Segmentation: ATE Masks Heterogeneity

In [ ]:
df_te_cr = pd.DataFrame(X_te_cr, columns=feat_cols)
df_te_cr['treatment'] = T_te_cr
df_te_cr['visit']     = Y_te_cr
df_te_cr['ite_pred']  = ite_x_cr

# Create segments using the uplift model's own output (not arbitrary cuts)
df_te_cr['uplift_quartile'] = pd.qcut(df_te_cr['ite_pred'], q=4,
                                       labels=['Bottom 25%', '25–50%', '50–75%', 'Top 25%'])

seg_cr = df_te_cr.groupby('uplift_quartile').apply(
    lambda g: pd.Series({
        'n': len(g),
        'treatment_rate': g['treatment'].mean(),
        'visit_rate_ctrl': g.loc[g.treatment==0, 'visit'].mean(),
        'visit_rate_treat': g.loc[g.treatment==1, 'visit'].mean(),
        'observed_lift': g.loc[g.treatment==1,'visit'].mean() - g.loc[g.treatment==0,'visit'].mean()
                         if (g.treatment==0).sum() > 0 else 0,
        'avg_pred_uplift': g['ite_pred'].mean(),
    })
).round(5)

print('=== Segmentation by Predicted Uplift Quartile ===')
print(seg_cr.to_string())

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(4)
bars = ax.bar(x, seg_cr['observed_lift'] * 100, color=PALETTE[:4], alpha=0.9, width=0.5)
ax.axhline(0, color='#8b949e', lw=1)
overall_lift = (df_te_cr.loc[df_te_cr.treatment==1,'visit'].mean() -
                df_te_cr.loc[df_te_cr.treatment==0,'visit'].mean()) * 100
ax.axhline(overall_lift, color=PALETTE[4], lw=2, linestyle='--',
           label=f'Overall ATE = {overall_lift:+.3f}pp')
ax.set_xticks(x)
ax.set_xticklabels(['Bottom 25%\n(low uplift)', '25–50%', '50–75%', 'Top 25%\n(high uplift)'], fontsize=10)
ax.set_ylabel('Observed lift (visit rate, pp)', fontsize=12)
ax.set_title('ATE Masks Heterogeneity: Lift by Predicted Uplift Quartile\n'
             '(The average hides that most of the value is in the top quartile)', fontsize=13)
ax.legend()
for bar, val in zip(bars, seg_cr['observed_lift'].values * 100):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:+.3f}pp', ha='center', fontsize=9, color='#e6edf3')
ax.grid(True, axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('figures/04_criteo_segmentation.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## Summary

| | Synthetic (NB 03) | Criteo (NB 04) |
|---|---|---|
| Ground truth ITE | ✅ Known | ❌ Not available |
| Validation method | Qini vs. true ITE + Spearman r | Qini vs. random baseline |
| Treatment balance | 50/50 | 85/15 (imbalanced) |
| Scale | 100K | Up to 13M |
| X-Learner advantage | Demonstrated (known ground truth) | Hypothesized (handles imbalance) |

**Pipeline is identical** — the same code works on both datasets, which is exactly the point of the two-dataset strategy.

**Next:** `05_decision_memo.md` — the ship/no-ship recommendation.